In [4]:
# --------------------------------------------
# 0. 환경 준비
# --------------------------------------------
!pip install xgboost

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, classification_report
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from google.colab import drive

# --------------------------------------------
# 1. 데이터 불러오기
# --------------------------------------------
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/정보기/scaled.csv')

# --------------------------------------------
# 2. Class 열 NaN 처리
# --------------------------------------------
nan_count = df['Class'].isnull().sum()

if nan_count > 0:
    print(f"[경고] Class 열에 {nan_count}개의 NaN 값이 존재하여 해당 행을 제거합니다.")
    df.dropna(subset=['Class'], inplace=True)

# --------------------------------------------
# 3. X, y 분리
# --------------------------------------------
X = df.drop('Class', axis=1)
y = df['Class']

# --------------------------------------------
# 4. Train/Test Split
# --------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# --------------------------------------------
# 5. 스케일링
# --------------------------------------------
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --------------------------------------------
# 6. XGBoost 모델 학습
#    - 클래스 불균형 해결 (양·음성 희소성 대응)
# --------------------------------------------
pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)

xgb_model = xgb.XGBClassifier(
    max_depth=5,
    learning_rate=0.03,
    n_estimators=800,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=pos_weight,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)

xgb_model.fit(X_train_scaled, y_train)

# --------------------------------------------
# 7. threshold = 0.5 적용
# --------------------------------------------
proba = xgb_model.predict_proba(X_test_scaled)[:, 1]
pred = (proba >= 0.5).astype(int)

# --------------------------------------------
# 8. Precision, Recall 출력
# --------------------------------------------
print("\n================ XGBoost 성능 ================")
print("Precision:", precision_score(y_test, pred))
print("Recall:", recall_score(y_test, pred))
print("\n", classification_report(y_test, pred))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

================ XGBoost 성능 ================
Precision: 0.8541666666666666
Recall: 0.8367346938775511

               precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.85      0.84      0.85        98

    accuracy                           1.00     56962
   macro avg       0.93      0.92      0.92     56962
weighted avg       1.00      1.00      1.00     56962

